In [136]:
import pandas as pd

In [137]:
from pathlib import Path
import sys
PROJECT_ROOT = Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [138]:
from python.ingestion.config import RAW_DATA_DIR
print(RAW_DATA_DIR)

/Users/anshajupadhyay/Downloads/Analytics Enginner/sales-analytics-pipeline/data/raw


In [139]:
# Load data
from python.ingestion.loader import load_raw_data
raw_data = load_raw_data()

In [140]:
# Data Set overview
for name, data in raw_data.items():
    print(name)
    data.info()
    rows, columns = data.shape
    print(f"-Rows: {rows}\n-Columns: {columns}")
    print("-"*50)

customers
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB
-Rows: 99441
-Columns: 5
--------------------------------------------------
geolocation
<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-n

In [141]:
# Missing Values
for name, data in raw_data.items():
    print(name)
    missing_count = data.isnull().sum()
    missing_count = missing_count[missing_count > 0]
    if missing_count.empty:
        print('No missing values found');
        print('-'*50)
        continue
    missing_percentage = round(missing_count/data.shape[0]*100,2)
    df_missing = pd.DataFrame({"Missing count":missing_count,"Missing %": missing_percentage})
    print(df_missing)
    print('-'*50)

customers
No missing values found
--------------------------------------------------
geolocation
No missing values found
--------------------------------------------------
order_items
No missing values found
--------------------------------------------------
order_payments
No missing values found
--------------------------------------------------
order_reviews
                        Missing count  Missing %
review_comment_title            87656      88.34
review_comment_message          58247      58.70
--------------------------------------------------
orders
                               Missing count  Missing %
order_approved_at                        160       0.16
order_delivered_carrier_date            1783       1.79
order_delivered_customer_date           2965       2.98
--------------------------------------------------
products
                            Missing count  Missing %
product_category_name                 610       1.85
product_name_lenght                   610 

In [142]:
# Duplicate Values
for name, data in raw_data.items():
    print(name)
    print(f"Number of duplicate rows are {data.duplicated().sum()} that is {round(data.duplicated().sum()/data.shape[0]*100,2)}% of total rows")
    print('-'*50)

customers
Number of duplicate rows are 0 that is 0.0% of total rows
--------------------------------------------------
geolocation
Number of duplicate rows are 261831 that is 26.18% of total rows
--------------------------------------------------
order_items
Number of duplicate rows are 0 that is 0.0% of total rows
--------------------------------------------------
order_payments
Number of duplicate rows are 0 that is 0.0% of total rows
--------------------------------------------------
order_reviews
Number of duplicate rows are 0 that is 0.0% of total rows
--------------------------------------------------
orders
Number of duplicate rows are 0 that is 0.0% of total rows
--------------------------------------------------
products
Number of duplicate rows are 0 that is 0.0% of total rows
--------------------------------------------------
sellers
Number of duplicate rows are 0 that is 0.0% of total rows
--------------------------------------------------
product_category_name_translation


In [143]:
# Unique Values
for name, data in raw_data.items():
    print(name)
    rows, columns = data.shape
    print(f"Unique values per column \n{data.nunique()} in {rows} rows")
    print('-'*50)

customers
Unique values per column 
customer_id                 99441
customer_unique_id          96096
customer_zip_code_prefix    14994
customer_city                4119
customer_state                 27
dtype: int64 in 99441 rows
--------------------------------------------------
geolocation
Unique values per column 
geolocation_zip_code_prefix     19015
geolocation_lat                717363
geolocation_lng                717615
geolocation_city                 8011
geolocation_state                  27
dtype: int64 in 1000163 rows
--------------------------------------------------
order_items
Unique values per column 
order_id               98666
order_item_id             21
product_id             32951
seller_id               3095
shipping_limit_date    93318
price                   5968
freight_value           6999
dtype: int64 in 112650 rows
--------------------------------------------------
order_payments
Unique values per column 
order_id                99440
payment_sequentia

In [144]:
# Data Values
for name, data in raw_data.items():
    print(name)
    print(data.describe().T)
    print('-'*50)

customers
                            count          mean           std     min  \
customer_zip_code_prefix  99441.0  35137.474583  29797.938996  1003.0   

                              25%      50%      75%      max  
customer_zip_code_prefix  11347.0  24416.0  58900.0  99990.0  
--------------------------------------------------
geolocation
                                 count          mean           std  \
geolocation_zip_code_prefix  1000163.0  36574.166466  30549.335710   
geolocation_lat              1000163.0    -21.176153      5.715866   
geolocation_lng              1000163.0    -46.390541      4.269748   

                                     min           25%           50%  \
geolocation_zip_code_prefix  1001.000000  11075.000000  26530.000000   
geolocation_lat               -36.605374    -23.603546    -22.919377   
geolocation_lng              -101.466766    -48.573172    -46.637879   

                                      75%           max  
geolocation_zip_code_prefi

In [ ]:
# Table Geolocation data 
'''
1. There are 261831 Number of duplicate rows, which is 26.18% of total rows
2. No missing values
2. Even after dropping duplicates, lat, long and zip code does not account for unique key
3. The only possible combination for finding a unqiue key is using this allc('geolocation_zip_code_prefix' ,
                                                                             'geolocation_lat' , 'geolocation_lng',\
                                                                             'geolocation_city','geolocation_state')
'''
geo_data = raw_data['geolocation']
geo_data.drop_duplicates(inplace= True)
print(geo_data[['geolocation_zip_code_prefix' ,'geolocation_lat' , 'geolocation_lng',\
                 'geolocation_city','geolocation_state']].duplicated().sum())


0


In [ ]:
# Order Items data 
'''
1. No duplciate 
2. No missing values in the data
3. Order_id, order_item_id, product_id,  seller_id none is completely unique
4. Since one order can have multiple items \
    order_id (str), order_item_id (int) combination provide as unique composition, that can be used a primary key
5. shipping_limit_date is a str
6. 
'''
order_items_data = raw_data['order_items']
order_items_data[['order_id','order_item_id']].duplicated().sum()

np.int64(0)

In [197]:
#Order payments
'''
No missing value
No duplicate value
order_id has duplicate values
order_id + payment_sequential create a composite primary key, beacuse a customer may pay an order \
    with more than one payment method. If he does so, a sequence will be created to so order_id and payment_sequential
    creates composite primary key
'''
order_payements_data = raw_data['order_payments']
order_payements_data[['order_id','payment_sequential']].duplicated().sum()

np.int64(0)

In [221]:
#Order reviews
'''
- High number in review_comment_message, review_comment_title
order_reviews
                        Missing count  Missing %
review_comment_title            87656      88.34
review_comment_message          58247      58.70
- No duplicate rows 
- Review id is not unique , since one review_id can be associated to multiple order ids
- Review id and order_id is a composite primary key
- review_creation_date and review_answer_timestamp are str and should be converted during transformation
'''
order_reviews_data = raw_data['order_reviews']
order_reviews_data[['review_id','order_id']].duplicated().sum()

np.int64(0)

In [ ]:
#Orders 
'''
- Tell about that order status 
- orders has few missing value
                               Missing count  Missing %
order_approved_at                        160       0.16
order_delivered_carrier_date            1783       1.79
order_delivered_customer_date           2965       2.98

- No duplicate rows
- Order id is unique, Customer_id is also unique
- All date columns are str and should be converted to date time during transformation
- Order_id can be parimary key and Customer_id can be unqiue key
- Customer_id act as foregin key reference from customer table
'''
orders_data = raw_data['orders']
orders_data['order_id'].duplicated().sum()

np.int64(0)

np.int64(0)

In [ ]:
#Orders 
'''
products_data
- products have few missing value 
                            Missing count  Missing %
product_category_name                 610       1.85
product_name_lenght                   610       1.85
product_description_lenght            610       1.85
product_photos_qty                    610       1.85
product_weight_g                        2       0.01
product_length_cm                       2       0.01
product_height_cm                       2       0.01
product_width_cm                        2       0.01

- No duplicate values
- 
'''
products_data = raw_data['products']
products_data['product_id'].duplicated().sum()

np.int64(0)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
